# Problem 3. PreTraining

Read the paper ”Training Compute-Optimal Large Language Models” Hoffmann et al. [2022] carefully and answer the following questions. For each question, provide specific evidence and citations from the paper to support your answer.

Note: Your answers should be specific and include relevant numerical evidence where appropriate. For each answer, clearly indicate which sections or tables of the paper you are referencing to support your arguments

1. **(5 points)** The paper presents three different approaches to determine the optimal trade-off between model size and number of training tokens. For Approach 2 (IsoFLOP profiles), what were the exponents $a$ and $b$ found for the relationships $N_{opt} \propto C^a$ and $D_{opt} \propto C^b$? How do these values differ from Kaplan et al.'s findings, and what is the practical implication of this difference for training large language models?

**Answer:**

For **Approach 2 (IsoFLOP profiles)**, the paper finds

$$
N_{\mathrm{opt}} \propto C^{0.49}, \qquad D_{\mathrm{opt}} \propto C^{0.51}.
$$

So the exponents are

$$
a = 0.49, \qquad b = 0.51.
$$

These values imply that, under compute-optimal scaling, **model size and training tokens should grow at nearly the same rate as compute increases**. This is very different from the earlier Kaplan et al. result,

$$
N_{\mathrm{opt}} \propto C^{0.73}, \qquad D_{\mathrm{opt}} \propto C^{0.27},
$$

which suggested putting much more of the extra compute into **larger models** rather than into **more data**.

The practical implication is important. Under the Hoffmann et al. analysis, many large language models built using Kaplan-style scaling are **under-trained**: they are too large for the amount of data they see. In other words, for a fixed compute budget, it is usually better to train a **smaller model on many more tokens** than to train a much larger model on a relatively small dataset. This is exactly the paper's central conclusion and explains why Chinchilla-style models outperform larger but data-starved models.  

**Reference:** IsoFLOP scaling discussion in the paper; comparison against Kaplan et al. is summarized in the PDF notes.


2. **(5 points)** For a given compute budget of $576 \times 10^{23}$ FLOPs (same as Gopher), what is the optimal model size and number of training tokens according to the paper's analysis? Compare this to Gopher's actual configuration.

**Answer:**

For the **Gopher compute budget**,

$$
C = 5.76 \times 10^{23} \text{ FLOPs},
$$

the paper's analysis (reported in **Table 3**) predicts an approximately compute-optimal configuration of:

- **Optimal model size:** about **67B parameters**
- **Optimal training tokens:** about **1.5T tokens**

Gopher itself was trained with:

- **280B parameters**
- **300B tokens**

So relative to the paper's compute-optimal estimate, Gopher was:

$$
\frac{280}{67} \approx 4.2\times
$$

about **4 times larger than optimal**, and

$$
\frac{1.5\text{T}}{0.3\text{T}} = 5
$$

trained on about **5 times fewer tokens than optimal**.

This is the exact type of mismatch Hoffmann et al. argue against. The paper validates this point with **Chinchilla** (roughly **70B parameters** trained on **1.4T tokens**), which outperforms Gopher despite being much smaller. That result strongly supports the conclusion that Gopher was **over-parameterized and under-trained** for its compute budget.  

**Reference:** **Table 3** and the paper's comparison between **Gopher** and **Chinchilla**.


3. **(5 points)** The paper estimates compute-optimal scaling using three distinct approaches: (1) fixing model sizes and varying training tokens to locate the loss minimum, (2) training multiple models at fixed compute budgets (IsoFLOP profiles), and (3) fitting a parametric loss function $L(N, D)$ to all runs jointly. Compare these three approaches in terms of their methodology and underlying assumptions. Why does the paper use all three rather than relying on a single method? Which approach do you find most methodologically convincing, and why?

**Answer:**

The paper uses **three complementary approaches** to estimate compute-optimal scaling.

### 1. Training curve envelope
This approach fixes a model size and varies the number of training tokens, then looks for the point where loss is minimized along the training curve.

**Methodology:** For each model size, trace how the loss changes as training proceeds and identify the loss minimum.  
**Assumption:** The minimum can be estimated reliably from the observed training curves, which may require interpolation or smoothing.

**Strength:** It uses the **full training trajectory**, not just the endpoint.  
**Limitation:** It depends on how the curve minimum is estimated and is therefore somewhat sensitive to interpolation choices.

### 2. IsoFLOP profiles
This approach fixes the total compute budget and trains multiple models of different sizes under that same budget.

**Methodology:** Compare final losses across models trained at equal compute, then identify which model size is best.  
**Assumption:** Final loss at a fixed compute budget is the right quantity for locating the optimum.

**Strength:** It is the most **direct** way to answer the compute-allocation question.  
**Limitation:** It uses only the **final loss values**, so it does not exploit the full shape of the training curves.

### 3. Parametric loss modeling
This approach fits a joint functional form over all runs:

$$
L(N, D) = E + \frac{A}{N^{\alpha}} + \frac{B}{D^{\beta}}.
$$

**Methodology:** Use all training runs together and estimate the parameters of a global scaling-law model.  
**Assumption:** The chosen functional form is a good approximation to how loss depends on parameter count and data.

**Strength:** It gives an **analytic model** of the relationship between loss, model size, and dataset size, and it uses all the data jointly.  
**Limitation:** It is the most model-dependent approach because it relies on the correctness of the assumed equation.

### Why use all three?
The paper uses all three because each method has different assumptions and different failure modes. If all three methods, despite their different viewpoints, point to similar scaling behavior, then the result is much more credible than if it came from only one technique. The agreement across methods is therefore part of the evidence.

### Most convincing approach
I find **Approach 2 (IsoFLOP profiles)** the most methodologically convincing because it most directly matches the research question: **for a fixed compute budget, what model size performs best?** It requires fewer structural assumptions than the parametric fit and is less dependent on curve interpolation than the training-envelope method. So although the other two are valuable for triangulation, IsoFLOP profiles provide the clearest and most direct evidence.  

**Reference:** The paper's three-method comparison, including the loss model and **IsoFLOP** analysis.


4. **(5 points)** Meta's LLaMA-3 70B model was trained on approximately 15 trillion tokens. Using the approximation $C \approx 6ND$ FLOPs (where $N$ is the number of parameters and $D$ is the number of training tokens), estimate the total compute budget used for LLaMA-3 70B. Then apply the Chinchilla scaling law ($N_{opt} \propto C^{0.49}$, $D_{opt} \propto C^{0.51}$, with the compute-optimal token-to-parameter ratio of $\approx 20$ tokens per parameter) to determine whether LLaMA-3 70B is over-parameterized, under-trained, or compute-optimal by Chinchilla standards. What does your analysis reveal about how the field's training priorities have shifted since the Chinchilla paper was published in 2022?

**Answer:**

For **LLaMA-3 70B**, we are given:

$$
N = 70 \times 10^9, \qquad D = 15 \times 10^{12}.
$$

Using the approximation

$$
C \approx 6ND,
$$

we get

$$
C \approx 6 \cdot (70 \times 10^9) \cdot (15 \times 10^{12})
   = 6.3 \times 10^{24} \text{ FLOPs}.
$$

Now apply the Chinchilla-style compute-optimal relationship. A useful rule of thumb from the paper is that the compute-optimal regime is close to **20 training tokens per parameter**.

If we kept the model at **70B parameters**, the Chinchilla-style optimal token count would be roughly

$$
D_{\mathrm{opt}} \approx 20N = 20 \times 70\text{B} = 1.4\text{T tokens}.
$$

But LLaMA-3 70B was trained on

$$
15\text{T tokens},
$$

which is more than

$$
\frac{15}{1.4} \approx 10.7
$$

times larger than that Chinchilla-style estimate.

So by **Chinchilla standards**, LLaMA-3 70B is **under-parameterized / over-trained on data**, not over-parameterized. It uses far more data than the token budget that would have been considered compute-optimal under the 2022 scaling law.

We can also estimate the Chinchilla-style optimal model size for this compute budget using the approximate relation

$$
C \approx 120N^2
$$

which comes from combining

$$
C \approx 6ND
$$

with

$$
D \approx 20N.
$$

Then

$$
N_{\mathrm{opt}} \approx \sqrt{\frac{C}{120}}
= \sqrt{\frac{6.3 \times 10^{24}}{120}}
\approx 2.29 \times 10^{11}
\approx 229\text{B parameters}.
$$

So a Chinchilla-style compute-optimal model at this compute level would be closer to **229B parameters**, not **70B**.

### Interpretation
This reveals a major shift in training priorities since 2022. Chinchilla emphasized **compute-optimality**: balance parameters and data carefully. More recent frontier practice, as illustrated by LLaMA-3 70B, appears to prioritize **very large-scale dataset exposure** even when it moves away from classic Chinchilla-optimal scaling. That suggests the field increasingly values broader coverage, robustness, and downstream generalization from more data, rather than strictly optimizing next-token loss under a fixed compute law.


5. **(5 points)** According to the paper's analysis, what are the implications for training a 1 trillion parameter model? Would this be compute-optimal with current practices? Explain using evidence from Table 3 of the paper.

**Answer:**

According to **Table 3** of Hoffmann et al., a **1 trillion parameter** compute-optimal model would require approximately:

$$
1.27 \times 10^{26} \text{ FLOPs}
$$

and about

$$
21.2 \text{ trillion training tokens}.
$$

This is an enormous requirement. To compare it with **Gopher's** training budget,

$$
C_{\mathrm{Gopher}} = 5.76 \times 10^{23} \text{ FLOPs},
$$

we compute:

$$
\frac{1.27 \times 10^{26}}{5.76 \times 10^{23}}
= \frac{1.27}{5.76} \times 10^3
\approx 220.5.
$$

So the exact **Table 3** value implies that a 1T model would need roughly **220 times the Gopher compute budget**.

The broader conclusion in the paper is that, unless one has a budget on the order of

$$
10^{26} \text{ FLOPs},
$$

a dense **1T-parameter** model is **not** compute-optimal. In practice, current or conventional training setups do **not** make such a model optimal, because both the **compute requirement** and the **data requirement** are too large.

The data side is just as important as the compute side. A 1T model would need about

$$
21.2\text{T tokens},
$$

which is vastly beyond the few-hundred-billion-token regime used by many earlier frontier models. So under the paper's analysis, training a 1T dense model with contemporary-style token budgets would make it severely **under-trained**.

### Conclusion
No, a **1 trillion parameter dense model would not be compute-optimal under current practices** unless one also scaled both compute and data to the extreme levels indicated in **Table 3**. The paper's broader message is that future scaling is bottlenecked not just by parameter count, but also by the ability to supply enough **training compute** and enough **high-quality tokens** to stay on the compute-optimal frontier.  

**Reference:** **Table 3** and the discussion following it.
